# Gemma v7 emit pilot (slot budget, quota-free)
Emits dict-coded IRs (ir-v2 book, v7 prompt, bare wire) with the model's own slot-budget repair loop. Run 031+.

In [ ]:
import subprocess
import sys


def pip_install(*args):
    print('+', 'pip install', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                     check=True)


pip_install('--upgrade', 'pip')
pip_install('llama-cpp-python')
# Server runtime closure (bare install lacks it; v3/v2 died on sse_starlette).
pip_install('llama-cpp-python[server]')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('huggingface_hub', 'hf_transfer')
import llama_cpp
import sse_starlette, starlette_context, fastapi, uvicorn  # noqa: F401
from llama_cpp.server.app import create_app  # noqa: F401
print('server closure OK (import chain verified pre-download)')
print('llama-cpp-python OK (CPU build)')


In [ ]:
import json as _json
import sys as _sys
import urllib.request as _urlreq

TUNNEL = ""  # stamped by phonehome/push.sh (public receiver URL); empty = disabled
TOKEN = ""  # stamped by phonehome/push.sh (bearer token); empty = disabled
KID = 'cpu-emit-gemma'


def hb(phase, msg=''):
    if not TUNNEL or not TOKEN:
        return
    try:
        body = _json.dumps({'kernel': KID, 'phase': phase, 'msg': str(msg)[:300]}).encode()
        req = _urlreq.Request(f'{TUNNEL}/hb?token={TOKEN}', data=body, headers={'Content-Type': 'application/json'})
        with _urlreq.urlopen(req, timeout=5):
            pass
    except Exception:
        pass


_prev_hook = _sys.excepthook


def _hook(t, v, tb, _p=_prev_hook):
    hb('fatal', f'{t.__name__}: {v}')
    return _p(t, v, tb)


_sys.excepthook = _hook
print('phone-home:', 'ON' if TUNNEL and TOKEN else 'OFF (unstamped)')

import json
import os
import re
import subprocess
import sys
import time
import urllib.request

REPO = 'unsloth/gemma-4-12b-it-GGUF'
GGUF_FILE = 'gemma-4-12b-it-Q6_K.gguf'
N_CTX = 2048
PORT = 8086
TEMP = 0
MAX_TOKENS = 256
BUDGET = 4

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

PROMPT_HEAD = '# IR emission prompt v7 (dict-coded, bare wire, tight slots)\n\nPinned: 2026-11-14 (v6 + hard 4-word slot budget with repair).\nTuning this prompt requires re-running the pilot, not edits.\n\nConvert the support reply below into minimal telegraphic IR: the complete\nanswer in the fewest lines. USE THE CODEBOOK for every line it covers,\npreferring multi-fact move codes (al-at) when one line carries several facts.\n\nCodebook (emit codes, never write out a coded line):\na means "Eligible for refund: {reason}"\nb means "Support hours: {hours}"\nc means "Support phone: {phone}"\nd means "Live Chat: {url}"\ne means "Overnight Shipping: {detail}"\nf means "Policies vary by {scope}"\ng means "Visit {url}"\nh means "Select {target}"\ni means "Standard Shipping: {time}"\nj means "Standard shipping to {dest} takes {time}"\nk means "Log in to account"\nl means "Pickup available at {loc}"\nm means "Pickup available"\nn means "Sign in to {portal}"\no means "Open {a} or {b}"\np means "Need {item}"\nq means "Open account {area}"\nr means "Complaint: {topic}"\ns means "Expedited Shipping: {time}"\nt means "Unsubscribe: click {target} at {where}"\nu means "Locate {doc} {id}"\nv means "Find {option}"\nw means "Provide {a} or {b}"\nx means "Website: {url}"\ny means "Shipping methods: {list}"\nz means "Provide {doc} for {issue}"\naa means "Follow prompts to {goal}"\nab means "Sign in with {creds}"\nac means "Refund: {amount}"\nad means "Serve {scope}"\nae means "Email: {email}"\naf means "Agent will {action}"\nag means "Enter {credential}"\nah means "Check {where} for {what}"\nai means "Pay via: {methods}"\naj means "Specify {what}"\nak means "Describe {problem}"\nal means "Eligible for refund: {r1} and {r2}"\nam means "Support hours: {h}; phone: {p}; chat: {u}"\nan means "Support phone: {p}; chat: {u}"\nao means "Visit {url}, select {target}"\nap means "Select {t1}, then {t2}"\naq means "Select {target}, follow prompts to {goal}"\nar means "Support hours: {h}; phone: {p}"\nas means "Expedited: {t1}; overnight: {t2}"\nat means "Eligible for refund: {r}; note {policy}"\nEmit one code per line as <code>|<slot1>|<slot2> with slot values in template order; slotless codes as <code> alone. Example: a|duplicate charges. Never add a : prefix.\n\nRules:\n\n1. One fact per line. No greeting, no filler, no full sentences unless needed.\n2. If a codebook entry covers the fact, emit ONLY its code line — never write the line out.\n3. Copy every number, date, name, URL, and quoted string EXACTLY (inside slots too).\n4. Keep every fact needed to answer fully. Drop only pleasantries.\n5. Plain lines, no bullets, no headers, no code fences, no markdown emphasis.\n6. Copy {{...}} markers character-exact; never fill them in or drop them.\n   (Synthetic-corpus rule: production has real values instead of markers.)\n7. Uncoded lines: ultra-terse fragments (2-6 words), never full sentences.\n8. Slot hygiene: no trailing |, no {} or slot-name words inside values, no\n   | inside values. If you lack a slot value, omit the whole line.\n9. Tight slots: at most 4 words per slot value. Slots carry VALUES ONLY\n   (numbers, names, URLs, markers, short labels) — never sentences or\n   explanations. Move extra words out of slots into uncoded lines, or\n   drop them if they are filler. Example: slot "further instructions\n   on how to reset your PIN" (8 words, REJECTED) becomes "PIN reset\n   steps" (3 words).\n\nThe runner appends:\n\n---\n\nREPLY TO COMPRESS:\n<text>\n\nIR:\n'
REFS = json.loads('[{"key": "bitext-10410", "reference": "I\'m thrilled that you\'ve decided to create an account with us! This is a great way to take advantage of our services and benefits. To get started, could you please provide me with your full name, email address, and a preferred username? Once I have this information, I\'ll guide you through the account creation process step by step. Rest assured, your privacy and security are of the utmost importance to us, and we\'ll ensure that your data is protected every step of the way. Let\'s begin this exciting journey together!"}, {"key": "bitext-15158", "reference": "Fear not, for I shall guide you on the noble path to obtain the revered document you seek - invoice {{Invoice Number}}. Embrace the power of my assistance as we navigate the intricacies of invoice retrieval, ensuring your satisfaction and triumph in this endeavor. Let us embark on this journey together!"}, {"key": "bitext-13235", "reference": "We understand your frustration and eagerness to track the progress of your shipment. Please provide us with the necessary details like the {{Tracking Number}} or {{Order Number}} so that we can check the status and estimated arrival time of your article. We apologize for any inconvenience caused and appreciate your patience as we work to resolve this issue."}, {"key": "bitext-9043", "reference": "I see what you mean! I\'m on the same page that you would like to speak with someone. Your request is important, and I\'m here to help. Let me quickly connect you with a human representative who will be able to assist you further. Kindly stay on the line while I transfer you."}, {"key": "bitext-15134", "reference": "I completely understand your need to retrieve the bill with the number #{{Invoice Number}}. Let me assist you with that. To access your bill, you can go to your account dashboard and navigate to the \\"Invoices\\" section. From there, you should be able to locate and download the bill with the number #{{Invoice Number}}. If you encounter any difficulties, please don\'t hesitate to let me know. I\'m here to help make this process as seamless as possible for you."}, {"key": "bitext-10406", "reference": "That\'s wonderful to hear! You\'re taking a great step towards accessing the exclusive benefits of a {{Account Category}} account with us. To get started, I\'ll guide you through the process. Could you please provide me with your full name, email address, and a preferred username? Once I have these details, I\'ll assist you in creating your {{Account Category}} account, ensuring a smooth and seamless experience for you. And remember, your privacy and security are of utmost importance to us, so you can trust that your information is in safe hands."}, {"key": "bitext-23979", "reference": "Definitely! I\'m thrilled to assist you in upgrading to our exceptional {{Account Type}} account:\\n\\n1. Sign in: Start by logging into your account on our platform.\\n2. Account Settings: Once you\'re logged in, navigate to your \'{{Settings}}\' or \'{{Profile}}\' section.\\n3. Upgrade Option: Within the account settings, look for an option called \'{{Upgrade Account}}\' or \'{{Account Change}}\'.\\n4. Select {{Account Type}}: Choose the \'{{Account Type}}\' option from the list.\\n5. Finalize the Switch: Follow the on-screen instructions to complete the transition successfully.\\n\\nIf you encounter any difficulties during the process or if you have any additional questions, please don\'t hesitate to reach out. I\'m here to ensure a seamless upgrade experience for you. Get ready to unlock the exclusive benefits of the {{Account Type}} account!\\""}, {"key": "bitext-26471", "reference": "First and foremost, I understand your anticipation for the refund of {{Currency Symbol}}{{Refund Amount}}. Your patience is truly appreciated as we work diligently to provide you with the latest updates. Allow me to quickly review the status of your refund and provide you with the information you need. While I gather the details, may I ask if you have received any email notifications or updates regarding your refund? Your insights will help us ensure a smooth resolution to this matter. Thank you for trusting us to address your concerns and for your understanding throughout this process."}, {"key": "bitext-17686", "reference": "I\'m sensitive to the fact that you\'re looking for assistance with unsubscribing from our newsletter. We appreciate your interest and we\'re sorry to see you go. To unsubscribe, you can simply click on the \\"Unsubscribe\\" link at the bottom of any newsletter email you receive from us. Alternatively, you can visit our website and navigate to the newsletter subscription page where you can manage your preferences and unsubscribe. If you encounter any difficulties or have any further questions, please don\'t hesitate to let me know."}, {"key": "bitext-13221", "reference": "We understand that you are eager to receive updates on your shipment and we apologize for any inconvenience caused. To provide you with an accurate estimation of when you can expect your shipment, could you please provide us with the {{Tracking Number}} or {{Order Number}}? With this information, we will be able to track your package and give you a precise delivery date. We appreciate your patience and cooperation in assisting us in resolving this matter."}, {"key": "bitext-17650", "reference": "Thank you for expressing interest in signing up for our newsletter! To subscribe, you can visit our website at {{Website URL}} and look for the newsletter sign-up form. The form usually appears on the homepage, sidebar, or footer of our website. You can provide your name and email address to complete the subscription process. We appreciate your enthusiasm, and we look forward to keeping you informed with our latest updates and promotions."}, {"key": "bitext-17646", "reference": "Thank you for contacting to us about canceling your subscription to our company newsletter. I understand that you would like to opt-out from receiving our newsletters. To cancel your subscription, please provide me with the email address you used to subscribe so that I can locate your account and proceed with the cancellation. Rest assured, your privacy and preferences are of utmost importance to us, and we will promptly honor your request."}, {"key": "bitext-22614", "reference": "I\'ll get right on it! I appreciate your interest in leaving a comment about our company. To provide your valuable feedback, you can send an email to our dedicated feedback address, which is {{Feedback Email}}. Our team is eager to hear your thoughts, experiences, and suggestions. Feel free to share your feedback with us at any time. We truly value your input!"}, {"key": "bitext-740", "reference": "I can see that canceling purchase {{Order Number}} is causing you some concern. Rest assured, I\'m here to assist you throughout the process. Let\'s get started:\\n\\n1. Log into Your Account: Start by signing in to your {{Online Company Portal Info}} using your credentials.\\n2. Access Your Orders: Once you\'re logged in, navigate to the \'{{Online Order Interaction}}\' or \'{{Online Order Interaction}}\' section.\\n3. Find Your Purchase: Look for the purchase associated with the order number {{Order Number}} and click on it.\\n4. Initiate Cancellation: In the purchase details, you\'ll find the option to \'{{Online Order Interaction}}\'. Please select this.\\n5. Confirm and Complete: The system might prompt you for final confirmation or ask for feedback. Please provide the necessary information as requested.\\n\\nIf you encounter any difficulties or have further questions, please don\'t hesitate to contact us. Our team is available during {{Customer Support Hours}} at {{Customer Support Phone Number}} or via the Live Chat feature on our {{Website URL}}. We appreciate your trust, and we\'re committed to ensuring your satisfaction throughout this process."}, {"key": "bitext-22622", "reference": "Sure! I\'m here to assist you in leaving a comment. Our goal is to make the process as smooth and effortless as possible. To leave a comment, you can follow these simple steps:\\n\\n1. Visit our website and navigate to the product page of the item you want to comment on.\\n2. Scroll down to the comment section located below the product description.\\n3. Click on the \\"Add Comment\\" button.\\n4. A comment box will appear where you can type your comment. Feel free to share your thoughts, feedback, or suggestions about the product.\\n5. Once you have composed your comment, click on the \\"Submit\\" button to post it.\\n\\nRemember, your comments are highly valued as they help us improve our products and provide better experiences for all our customers. If you have any specific questions or require further assistance, please don\'t hesitate to let me know."}, {"key": "bitext-1958", "reference": "Happy to get your message to us regarding your request to add some items to purchase number {{Order Number}}. We understand that you would like to modify your order, and we\'re here to assist you with that. To ensure we get this right, could you please provide us with the specific items you\'d like to add? We\'ll make sure to make the necessary adjustments to your purchase and provide you with an updated order summary. Your satisfaction is our top priority, so please don\'t hesitate to let us know if you have any other questions or concerns."}, {"key": "bitext-17634", "reference": "I\'ve grasped that you\'re unsure about how to unsubscribe from our corporate newsletter. I\'m here to guide you through the process. To unsubscribe, you can simply click on the \\"Unsubscribe\\" link at the bottom of any newsletter email you receive from us. Alternatively, you can reply to the email and request to be removed from the mailing list. If you\'re encountering any difficulties with the unsubscribe process, please provide me with your email address, and I\'ll personally ensure your removal from our newsletter distribution. We appreciate your interest in our company updates, and if there\'s anything else I can assist you with, please don\'t hesitate to let me know."}, {"key": "bitext-9880", "reference": "We\'re here for you! I\'m aware that you would like to chat with a person and I\'m here to assist you. Our dedicated team of human representatives is available to help you with your query. To connect you with the right person, may I please ask for some details about the topic or issue you need assistance with? Your insights will enable us to provide you with personalized and efficient support. Please take a moment to share your concerns, and we will ensure that you are connected with the most suitable representative. We appreciate your cooperation and look forward to resolving your query promptly."}, {"key": "bitext-17666", "reference": "I\'ll make it happen! I\'d be thrilled to assist you with receiving our corporate newsletter. Staying connected with our company\'s latest updates, valuable insights, and exciting news is a fantastic way to stay informed and engaged. To ensure you receive our newsletter, could you please provide me with your email address? I\'ll make sure you\'re subscribed immediately. Rest assured, we prioritize the privacy and security of your information. We look forward to keeping you informed and inspired through our newsletter!"}, {"key": "bitext-17942", "reference": "I\'ve noticed that you\'re looking for the right place to inform us about any issues with your payments. You can reach out to our customer support team through various channels such as phone, email, or live chat. Our dedicated team is available around the clock to assist you with any payment concerns you may have. They will guide you through the process of reporting the issue and help resolve it as quickly as possible. We truly appreciate you bringing this to our attention as it allows us to continuously improve our payment systems and offer the best possible support to our valued customers."}, {"key": "bitext-18843", "reference": "I\'ve understood you\'re seeking assistance in resolving an error with your online payments. Rest assured, we\'re here to help you navigate through this issue and find a solution. Could you please provide more details about the error you\'re encountering? This will enable us to better understand the situation and provide you with the most relevant and effective guidance. Thank you for reaching out, and we look forward to resolving this error together."}, {"key": "bitext-19535", "reference": "Happy to hear from you! I truly understand that you need assistance with acquiring some items. I\'m here to help you find the perfect products that meet your needs and preferences. To get started, could you please provide me with more details about the items you are interested in? This will allow me to provide you with personalized recommendations and ensure a smooth and pleasant shopping experience for you."}, {"key": "bitext-21166", "reference": "Appreciate you reaching out to us to report errors with the sign-up process. We understand how frustrating it can be to encounter difficulties during registration, and we appreciate your proactive approach in notifying us. Your feedback is invaluable in helping us identify and resolve any issues. To assist you further, could you please provide specific details about the errors you encountered? This information will enable us to investigate and address the problem promptly. We assure you that we are committed to enhancing our sign-up process and ensuring a smooth experience for all users. Thank you for your cooperation and patience."}, {"key": "bitext-19529", "reference": "I see what you mean! I\'m on the same page that ordering products can sometimes be confusing, especially if you\'re new to the process. Don\'t worry, I\'m here to help you through it. To begin, could you please let me know which specific products you\'re interested in ordering? Once I have that information, I\'ll guide you step-by-step on how to place your order."}, {"key": "bitext-21170", "reference": "We understand that you\'re facing difficulties in notifying us about errors during the signup process, and we appreciate you reaching out for assistance. Our team is here to help you overcome these challenges and ensure a smooth registration experience. To better understand the specific errors you\'re encountering, could you please provide us with more details? This will enable us to offer tailored solutions and address any technical issues you may be facing. Your feedback is invaluable, and together we will work towards resolving this problem."}, {"key": "bitext-708", "reference": "I certainly recognize that you no longer want to proceed with your order with the order number {{Order Number}}. I\'m here to assist you in canceling it promptly and smoothly.\\n\\nTo cancel your order, please follow these steps:\\n\\n1. Sign in to Your Account: Access our platform by signing in to your {{Online Company Portal Info}}.\\n2. Locate Your Order: Navigate to the \'{{Online Order Interaction}}\' or \'{{Online Order Interaction}}\' section to find your order.\\n3. Select the Order: Look for the specific order with the order number {{Order Number}} and click on it.\\n4. Initiate the Cancellation: You should see an option labeled \'{{Online Order Interaction}}\' associated with your purchase. Please select this to begin the cancellation process.\\n5. Confirm the Cancellation: The system may ask you to confirm the cancellation. Follow the prompts and provide any necessary feedback.\\n\\nIf you encounter any difficulties or have any further questions, our dedicated team is available to assist you during {{Customer Support Hours}}. Feel free to reach out to us at {{Customer Support Phone Number}} or use the Live Chat feature on our {{Website URL}}.\\n\\nWe appreciate your understanding and apologize for any inconvenience caused."}, {"key": "bitext-22670", "reference": "We truly value your comment regarding our services! Your feedback is essential in helping us evaluate and enhance the customer experience we provide. To ensure we capture your insights effectively, please provide specific details about your experience or any areas you feel we can improve. Rest assured, your comment will be thoroughly reviewed and shared with the relevant teams. Thank you for taking the time to share your thoughts with us!"}, {"key": "bitext-14457", "reference": "I must commend your dedication to keeping the information on your {{Account Type}} account up to date. It shows how committed you are to maintaining accuracy and ensuring the best experience possible. To update the information on your {{Account Type}} account, you can easily access your account settings through our website or mobile app. From there, you\'ll be able to modify any details that may have changed. If you encounter any difficulties or have any questions along the way, our dedicated customer support team is here to assist you. They can guide you through the process and provide any necessary instructions. It\'s great to see your proactive approach in keeping your account information current. Together, we\'ll make sure your {{Account Type}} account meets all your needs and requirements."}, {"key": "bitext-6275", "reference": "Of course! I understand your desire to have a clear understanding of the situations in which you can request a refund. Allow me to provide you with a comprehensive breakdown of the most common cases:\\n\\n1. **Product/Service Defect:** If you receive a product or service that has a defect or doesn\'t match the description provided, you are generally eligible for a refund.\\n\\n2. **Cancellation within Grace Period:** If you change your mind and decide to cancel your purchase within the grace period specified by the vendor, you should be eligible for a refund.\\n\\n3. **Unauthorized Charges:** If you notice any charges on your account that you didn\'t authorize or recognize, it\'s important to bring it to our attention, and we will assist you in investigating and resolving the issue.\\n\\n4. **Event Cancellation:** If you purchased tickets for an event that gets canceled without a rescheduled date, you should be eligible for a refund.\\n\\n5. **Duplicate Charges:** In the event that you are charged more than once for the same item, please notify us immediately, and we will rectify the situation promptly.\\n\\n6. **Non-receipt of Goods:** If you haven\'t received your purchased product within the estimated delivery time frame, you may qualify for a refund.\\n\\nPlease note that refund policies may vary among different vendors and service providers. It is always recommended to review the specific refund policy of the vendor or consult their terms and conditions for more detailed information. If you have a specific order or concern, kindly provide us with the relevant details, such as the order number, and we will assist you further in resolving your query.\\n\\nI hope this information clarifies the cases in which you can request a refund. If you have any other questions or need further assistance, feel free to let me know!"}, {"key": "bitext-14427", "reference": "How delightful it is to witness your determination to edit the details included on your profile!I\'m here to guide you through this process and ensure a seamless and satisfactory experience. To begin, could you please provide me with more specific information about the details you would like to edit on your profile? This will enable me to assist you more effectively. Rest assured, I\'m fully committed to making this process as smooth as possible for you."}, {"key": "bitext-21106", "reference": "We understand that you\'re unsure of how to communicate a sign-up problem, and we\'re here to help. To notify us about any issues you\'re facing during the sign-up process, you can reach out to our customer support team via email, phone, or live chat. Our dedicated team is available to assist you and gather the necessary details to address the problem promptly. Rest assured, we value your feedback and prioritize resolving any sign-up issues you may encounter. Together, we can ensure a seamless sign-up experience."}, {"key": "bitext-21138", "reference": "Thank you for reaching out to us and informing us about the signup issues you\'re experiencing. We understand how frustrating it can be to encounter problems during the registration process, and we sincerely apologize for any inconvenience caused. Your feedback plays a crucial role in helping us improve our services, and we appreciate your patience and understanding as we work to resolve this issue for you. Could you please provide us with some specific details about the problems you encountered? This will assist us in investigating and addressing the issue promptly. We\'re here to assist you every step of the way and ensure that your registration experience is as smooth as possible."}, {"key": "bitext-4491", "reference": "No worries! I understand that you\'re unsure about the steps to take in order to locate the bill with the number #{{Invoice Number}}. Let me guide you through the process. First, could you please provide me with the name of the account holder or any other relevant information associated with the bill? This will help me assist you more efficiently."}, {"key": "bitext-21114", "reference": "Glad you contacted to us and expressing your concerns about a sign-up issue. We understand that you are unsure about the process of notifying us about this problem, and we\'re here to assist you every step of the way. To report a sign-up issue, you can either reach out to our customer support team via email, phone, or live chat, or you can submit a support ticket through our website. Whichever method you choose, providing specific details about the issue you encountered, such as error messages, account information, or any steps you took before experiencing the problem, would greatly aid our investigation and help us provide a swift resolution. We value your feedback and are committed to resolving issues promptly to ensure a smooth sign-up experience for all our users."}, {"key": "bitext-792", "reference": "I recognize that you\'ve recently made a purchase and now wish to cancel the order with the order number {{Order Number}}. I apologize for any inconvenience this may have caused. To proceed with the cancellation, please follow these steps:\\n\\n1. Sign in to Your Account: Access our platform by logging in to your {{Online Company Portal Info}}.\\n2. Find Your Order History: Once logged in, navigate to the \'{{Online Order Interaction}}\' or \'{{Online Order Interaction}}\' section.\\n3. Locate Your Purchase: Look for the specific purchase associated with the order number {{Order Number}} and click on it.\\n4. Initiate Cancellation: Within the purchase details, you should find an option labeled \'{{Online Order Interaction}}\'. Please select this option.\\n5. Confirm Cancellation: The system may request additional information or confirmation before finalizing the cancellation. Kindly provide any required details or follow the prompts to proceed.\\n\\nIf you encounter any difficulties or need further assistance during the cancellation process, please don\'t hesitate to reach out to our dedicated support team. They are available during {{Customer Support Hours}} at {{Customer Support Phone Number}} or through the Live Chat feature on our website at {{Website URL}}. Your satisfaction is our priority, and we\'re here to help you every step of the way."}, {"key": "bitext-16721", "reference": "I\'ve got that you need assistance with asking for restitutions of your money. It can be frustrating when you feel you\'ve been wronged and want to seek reimbursement. To help you with this process, I recommend reaching out to our customer support team. They have the expertise to guide you through the necessary steps and provide you with the relevant information. You can contact them at {{Customer Support Phone Number}} or through the Live Chat on our website at {{Website URL}}. They will be able to address your concerns and work towards resolving the issue with you. Rest assured, we\'re committed to ensuring your satisfaction and resolving any issues you may have."}, {"key": "bitext-16735", "reference": "I\'m fully aware of your frustration and urgency in seeking reimbursement for the product you purchased. Rest assured, I\'m here to assist you every step of the way. To ensure we address your concern promptly and accurately, I kindly request you to provide me with some more details regarding your purchase. Could you please share the order number or any other relevant information? This will enable us to locate your purchase and expedite the refund process for you. Your satisfaction is our priority, and I\'ll do my best to resolve this matter as quickly as possible."}, {"key": "bitext-12836", "reference": "Positively! I understand your curiosity about shipments to {{Delivery City}}. Let me provide you with the details:\\n\\n1. Standard Shipping: Typically arrives within 7-10 business days.\\n2. Expedited Shipping: Speedier option delivering in 3-5 business days.\\n3. Overnight Shipping: The fastest choice, guaranteeing next-day delivery.\\n4. In-Store Pickup: If you reside near our {{Delivery City}} store, you can opt to collect your items personally.\\n\\nThese options are available for shipments to {{Delivery City}}, ensuring that you receive your products conveniently and promptly. If you have any other questions or if there\'s anything else I can assist you with, please feel free to let me know. I\'m here to enhance your shipping experience!\\""}, {"key": "bitext-14487", "reference": "How exciting it is to witness your determination to modify the data on your profile!I\'m here to guide you through the process and provide you with any necessary assistance. To get started, could you please let me know which specific data you would like to modify on your profile? With this information, I\'ll be able to navigate you towards a seamless experience and ensure that all your requirements are met."}, {"key": "bitext-788", "reference": "I\'ve been informed that you require assistance with canceling your purchase of the item with the order number {{Order Number}}. I apologize for the inconvenience caused and I\'m here to help you navigate the cancellation process smoothly.\\n\\nTo cancel your purchase, please follow these steps:\\n\\n1. Sign in to your {{Online Company Portal Info}} using your login credentials.\\n2. Locate the section for your order history or recent purchases. This can usually be found under the \'{{Online Order Interaction}}\' or \'{{Online Order Interaction}}\' tab.\\n3. Look for the specific purchase with the order number {{Order Number}}.\\n4. Click on the purchase to access the order details.\\n5. Within the order details, you should find an option to cancel the purchase. It may be labeled as \'{{Online Order Interaction}}\' or something similar. Click on this option.\\n6. Follow any additional prompts or instructions to confirm the cancellation.\\n\\nIf you encounter any difficulties or have further questions, please don\'t hesitate to contact our dedicated customer support team. They are available during {{Customer Support Hours}} at {{Customer Support Phone Number}} or through the Live Chat feature on our website at {{Website URL}}. Your satisfaction is our top priority, and we are here to assist you every step of the way."}, {"key": "bitext-10594", "reference": "That\'s wonderful! I\'m thrilled to assist you in opening a {{Account Type}} account. To get started, could you please provide me with your full name, email address, and a preferred username? Once I have these details, I\'ll guide you through the process step by step, ensuring you\'re well-informed about the features and benefits of our {{Account Type}} account. Rest assured that your security and privacy are of utmost importance to us, and we\'ll take every measure to protect your information."}, {"key": "bitext-12820", "reference": "Assuredly! I\'m glad you\'re interested in exploring our shipping methods. Let me provide you with the available options we have for you:\\n\\n1. Standard Shipping: This is our regular shipping option, and it typically takes {{Date Range}} business days to deliver to your location in Mexico.\\n\\n2. Expedited Shipping: If you need your order a bit faster, our expedited shipping option is available, and it usually takes {{Date Range}} business days for delivery.\\n\\n3. Express Shipping: For those in need of urgent delivery, we offer express shipping, which delivers your order to Mexico within {{Date Range}} business days.\\n\\nWe strive to provide you with flexible shipping choices to meet your needs. If you have any specific requirements or questions about our shipping methods, feel free to let me know, and I\'ll be happy to assist you further."}, {"key": "bitext-13592", "reference": "We understand your eagerness to track the progress of your item and anticipate its arrival. To check the estimated delivery date, you can visit our website and navigate to the \\"Order Tracking\\" or \\"Track Your Shipment\\" section. There, you will be prompted to enter either the {{Tracking Number}} or {{Order Number}} associated with your purchase. By entering this information, you will gain access to real-time updates on the status and expected delivery date of your item. Should you encounter any difficulties during this process, our customer support team is available to assist you at {{Customer Support Phone Number}} or through our Live Chat service on our website at {{Website URL}}. We appreciate your patience and look forward to providing you with the information you need."}, {"key": "bitext-12816", "reference": "Sure! We offer delivery service to {{Delivery City}}, so you can conveniently receive your order in the City of Brotherly Love. We have a range of delivery options available, including standard shipping, expedited shipping, and even in-store pickup if you prefer. You can choose the option that best suits your needs and have your items delivered to your doorstep. If you have any specific concerns or questions, feel free to ask. We\'re here to ensure a seamless delivery experience for you in {{Delivery City}}!"}, {"key": "bitext-11830", "reference": "Grateful you reached out about canceling your {{Account Category}} account. I\'m here to assist you with the process. Could you please provide me with some additional details about the account you\'d like to remove? This will help me guide you through the necessary steps to ensure a smooth cancellation. Your satisfaction is our top priority, and we appreciate your patience as we work together to address your concerns."}, {"key": "bitext-2520", "reference": "We appreciate your inquiry about modifying your delivery address. To make any changes to your delivery address, please contact our customer service department at [customer service phone number]. Our dedicated team will assist you in updating your shipping information."}, {"key": "bitext-12844", "reference": "By all means, let\'s navigate the enchanting realm of delivery methods together! I\'m thrilled to guide you through an exploration of our marvelous delivery options.\\n\\nHere, behold the range of delivery methods we offer for your convenience:\\n\\n1. Standard Shipping: The classic choice, which typically arrives within {{Date Range}} business days. Perfect for those who savor the joy of anticipation.\\n\\n2. Expedited Shipping: If time is of the essence, this option gets your items to you in a swift {{Date Range}} business days.\\n\\n3. Overnight Shipping: For those moments of urgency, you can rely on this option to ensure next-day delivery.\\n\\n4. In-Store Pickup: Embrace convenience and personal connection with our In-Store Pickup option. Simply choose your nearest {{Store Location}} to retrieve your items at your leisure.\\n\\nNow, as you ponder which delivery method suits your needs best, know that I\'m here to assist with any questions or concerns that may arise. Together, let\'s weave a seamless delivery experience, tailored just for you!"}, {"key": "bitext-18199", "reference": "I\'m here to guide you through resolving any errors you\'re encountering with your payment. Rest assured that you\'re not alone in this, and we\'re committed to finding a solution together. To better assist you, could you please provide more details about the specific errors you\'re experiencing? The more information you share, the better equipped we are to pinpoint the issue and provide tailored assistance. Thank you for reaching out, and I appreciate your cooperation in troubleshooting the payment errors."}, {"key": "bitext-18161", "reference": "Grateful you reached out to us about the payment problem you\'ve encountered. To inform us of the issue, you can contact our customer support team through various channels such as phone, email, or live chat. Our dedicated representatives are available {{Customer Support Hours}} to assist you and gather all the necessary details regarding the payment problem. They will guide you through the process and ensure that your concerns are appropriately addressed. We value your feedback, and we appreciate your proactive approach in reporting this issue to us."}, {"key": "bitext-1653", "reference": "Thank you for contacting! I certainly recognize that you would like to add an item to order number {{Order Number}}. I\'m here to help you with that. Could you please provide me with the details of the item you wish to add? It could be the name, quantity, and any other relevant information. This way, I can assist you promptly in updating your order to meet your expectations. Your satisfaction is our top priority!"}, {"key": "bitext-18119", "reference": "I apologize for the frustration you\'re facing with the issue in your payments. I understand that you\'re seeking assistance in resolving this matter. Please provide me with more details about the problem you\'re encountering, and together we\'ll work towards finding a solution. Your feedback is crucial in helping us improve our services and address any concerns you may have. Thank you for reaching out, and we\'re here to support you every step of the way."}, {"key": "bitext-9136", "reference": "We\'re all ears! I\'m tuned into the idea that you are unsure about how to contact somebody. Let me guide you through the steps so that you can easily get in touch with one of our representatives. To contact somebody, you can try the following options:\\n\\n1. Phone: You can call our customer support team at {{Customer Support Phone Number}}. They are available {{Customer Support Hours}} to assist you and address any questions or concerns you may have.\\n\\n2. Live Chat: You can chat with one of our representatives in real-time by visiting our website at {{Website URL}}. Look for the chat icon and click on it to initiate a conversation.\\n\\n3. Email: You can also reach out to us via email at {{Customer Support Email}}. Simply compose your message and provide details about your inquiry or issue, and our team will respond as soon as possible.\\n\\n4. Social Media: If you prefer to connect through social media, you can send us a direct message on platforms like Facebook or Twitter. Our team will be happy to assist you there as well.\\n\\nI hope this clarifies the various ways in which you can contact somebody. If you have any further questions, please don\'t hesitate to let me know. We\'re here to help!"}, {"key": "bitext-5172", "reference": "I\'ll make it happen! I\'m glad to assist you in checking our accepted payment options. To find out what payment methods we accept, you can visit our website and navigate to the \\"Payment Methods\\" page. There you will find a comprehensive list of the payment options we currently support. If you have any specific questions or need further guidance, please don\'t hesitate to let me know. I\'m here to ensure you have all the information you need for a seamless payment experience."}, {"key": "bitext-5104", "reference": "Appreciate your message and expressing your need to locate our allowed payment options. To view the available payment methods, you can visit our website\'s payment page. On the website, navigate to the \\"Payment Methods\\" section or search for \\"Payment Options\\" in the search bar. This will lead you to a page where you can find a detailed list of our accepted payment methods. If you encounter any difficulties or have further questions, please don\'t hesitate to let me know. I\'m here to assist you every step of the way."}, {"key": "bitext-20427", "reference": "Absolutely! I\'m here to assist you in retrieving your user access key. It\'s essential to have access to your account, so let\'s go through the steps together:\\n\\n1. First, visit the \\"{{Login Page URL}}\\" of our platform.\\n2. Look for the option that says \\"{{Forgot Your Access Key}}\\" and click on it.\\n3. You may be prompted to provide some information about your account to verify your identity.\\n4. Once verified, you will receive an email containing instructions on how to retrieve your user access key.\\n5. If you don\'t find the email in your main inbox, please check your spam or junk folders just in case it ended up there.\\n\\nIf you encounter any difficulties or have further questions, feel free to let me know. I\'m here to ensure a smooth and hassle-free process for you!\\""}, {"key": "bitext-20447", "reference": "Oh, I understand that you\'re unsure about the process to recover your profile password. Don\'t worry, I\'m here to guide you through it. Here are the steps you can follow:\\n\\n1. First, visit the \\"{{Login Page URL}}\\" of our platform.\\n2. On the login page, look for the option that says \\"{{Forgot Password}}\\". It should be somewhere near the login form.\\n3. Once you click on the \\"Forgot Password\\" option, you\'ll be redirected to a password recovery page.\\n4. On the password recovery page, you\'ll be asked to provide your registered email address or username associated with your profile.\\n5. After submitting the required information, keep an eye on your email inbox. You should receive an email with detailed instructions on how to reset your password.\\n6. Just follow the instructions provided in the email, and you\'ll be able to set a new password for your profile.\\n\\nIf you encounter any difficulties during this process or have any further questions, please don\'t hesitate to reach out to me. I\'m here to help you every step of the way!\\""}, {"key": "bitext-1633", "reference": "I fathom that you need assistance in deleting a specific product from purchase number {{Order Number}}. I apologize for any inconvenience caused and assure you that I\'m here to help you resolve this issue. To proceed with the deletion, could you please provide me with the details of the product you wish to remove? This will allow us to take the necessary steps and ensure your purchase is exactly as you desire. Thank you for your cooperation in helping us improve our services."}, {"key": "bitext-19852", "reference": "Thank you for your inquiry! I understand that you\'re unsure about the process of purchasing our products. Don\'t worry, I\'m here to assist you every step of the way. To get started, may I ask if you have already selected the specific products you\'d like to purchase? If not, I can help you explore our wide range of products and provide recommendations based on your preferences and needs. Once we have identified the products you wish to buy, I\'ll guide you through the purchasing process, ensuring that it is simple and straightforward. Let\'s make sure your shopping experience with us is enjoyable and successful!"}, {"key": "bitext-20437", "reference": "Positively! I\'m here to assist you in recovering your PIN and restoring access to your account. Let\'s go through the process step by step:\\n\\n1. Start by navigating to the account recovery page on our platform.\\n2. Look for the \\"Forgot PIN\\" option and click on it.\\n3. You might be prompted to answer a few security questions to verify your identity. This is to ensure the privacy and security of your account.\\n4. Once your identity is verified, you will receive further instructions on how to reset your PIN through the email associated with your account.\\n5. Follow the instructions provided in the email carefully to set up a new PIN of your choice.\\n\\nIf you encounter any challenges or require additional assistance, don\'t hesitate to reach out. Remember, we are here to support you every step of the way!\\""}, {"key": "bitext-8243", "reference": "We\'re all ears! I\'m tuned into the idea that you\'re unsure about the steps to speak to our customer support team. Don\'t worry; I\'m here to guide you. To get in touch with our dedicated customer support, you can reach out to them during {{Customer Support Hours}}. They are available via phone at {{Customer Support Phone Number}} or through the Live Chat feature on our website at {{Website URL}}. By using any of these channels, our support team will be more than happy to assist you with any questions, concerns, or issues you may have. Remember, we are here to provide you with the support you need and make your experience with us as seamless as possible. How can I assist you further?"}, {"key": "bitext-14447", "reference": "How delightful it is to embark on this invigorating journey of rectifying and enhancing the details on your prestigious {{Account Type}} account!I\'m thrilled to provide you with my expertise and support to ensure that your account information is accurately updated and tailored to your preferences. Together, let\'s make your {{Account Type}} account shine even brighter. Please share the specific details you would like to correct, and I will be here to guide you every step of the way."}, {"key": "bitext-10546", "reference": "I appreciate your interest in opening a {{Account Category}} account with us! To begin the process, you can visit our website at {{Website URL}}. On the homepage, you will find a \\"Sign Up\\" or \\"Create Account\\" button. Click on that button to get started with the account creation process. If you encounter any difficulties or have any questions along the way, please don\'t hesitate to reach out to our customer support team. We are available {{Customer Support Hours}} and can be reached at {{Customer Support Phone Number}} or through the Live Chat on our website. We\'re here to assist you every step of the way in opening your {{Account Category}} account with us."}]')
ARITIES = json.loads('{"a": 1, "b": 1, "c": 1, "d": 1, "e": 1, "f": 1, "g": 1, "h": 1, "i": 1, "j": 2, "k": 0, "l": 1, "m": 0, "n": 1, "o": 2, "p": 1, "q": 1, "r": 1, "s": 1, "t": 2, "u": 2, "v": 1, "w": 2, "x": 1, "y": 1, "z": 2, "aa": 1, "ab": 1, "ac": 1, "ad": 1, "ae": 1, "af": 1, "ag": 1, "ah": 2, "ai": 1, "aj": 1, "ak": 1, "al": 2, "am": 3, "an": 2, "ao": 2, "ap": 2, "aq": 2, "ar": 2, "as": 2, "at": 2}')
NREFS = len(REFS)
assert NREFS >= 1, NREFS
print('emit set OK:', NREFS, 'refs (embedded)')
hb('eval-ok', f'{len(REFS)} refs')

hb('downloading', GGUF_FILE)
path = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print('downloaded:', path, flush=True)
hb('downloaded', GGUF_FILE)

log = open('/tmp/llama-server.log', 'w')
help_txt = subprocess.run([sys.executable, '-m', 'llama_cpp.server', '--help'], capture_output=True, text=True).stdout
for flag in ['--n_threads', '--n_threads_batch', '--n_batch', '--n_ubatch', '--type_k', '--type_v', '--use_mlock', '--flash_attn']:
    assert f' {flag}' in help_txt or f'\n{flag}' in help_txt, f'{flag} missing from server --help'
EXTRA = ['--n_threads', '4', '--n_threads_batch', '4', '--n_batch', '512', '--n_ubatch', '512',
         '--type_k', '8', '--type_v', '8', '--use_mlock', 'true', '--flash_attn', 'true']
print('tuned flags:', ' '.join(EXTRA), flush=True)
cmd = [sys.executable, '-m', 'llama_cpp.server',
       '--model', path, '--n_gpu_layers', '0',
       '--n_ctx', str(N_CTX), '--host', '127.0.0.1',
       '--port', str(PORT)] + EXTRA
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(60):
    if proc.poll() is not None:
        print('--- server died; log tail ---')
        print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
        print('--- env capsule ---')
        print(subprocess.run('free -g; df -h /tmp | tail -1', shell=True, capture_output=True, text=True).stdout)
        raise RuntimeError(f"server exited with code {proc.returncode}")
    time.sleep(10)
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
            print('READY, status', r.status, flush=True)
            hb('ready', 'gemma12-emit')
            break
    except Exception:
        if i % 6 == 0:
            print(f'waiting for server... ({i * 10}s)', flush=True)
else:
    raise RuntimeError('server did not come up in 10 min')

CODE_RE = re.compile(r'^(?::([a-z]{1,2})(?:\|(.*))?|([a-z]{1,2})\|(.*))$')


def _split_code(s):
    m = CODE_RE.match(s)
    if not m:
        return None
    if m.group(1) is not None:
        return m.group(1), m.group(2) or ''
    return m.group(3), m.group(4) or ''


def slot_violations(text, budget):
    out = []
    for i, ln in enumerate(text.split('\n')):
        s = ln.strip()
        parts = _split_code(s)
        if not parts:
            continue
        code, rest = parts
        for v in rest.split('|'):
            if len(v.split()) > budget:
                out.append((i + 1, code, len(v.split())))
    return out


def code_violations(text):
    out = []
    for i, ln in enumerate(text.split('\n')):
        s = ln.strip()
        parts = _split_code(s)
        if not parts:
            continue
        code, rest = parts
        if code not in ARITIES:
            out.append((i + 1, code, 'unknown', -1, -1))
            continue
        vals = rest.split('|')
        while vals and vals[-1] == '':
            vals.pop()
        if len(vals) != ARITIES[code]:
            out.append((i + 1, code, 'arity', ARITIES[code], len(vals)))
    return out


def repair_ask(prompt, ir, code, slots, budget):
    bullets = []
    for (ln, c, kind, want, got) in code:
        if kind == 'arity':
            want_txt = 'no slots' if want == 0 else f'exactly {want} slot' + ('s' if want != 1 else '')
            bullets.append(f"line {ln}: code '{c}' needs {want_txt}, got {got} -- fix the |-separated values to match the codebook entry (add, merge, or drop values; keep every entity).")
        else:
            bullets.append(f"line {ln}: unknown code '{c}' (not in the codebook) -- use a codebook code or a plain-text line.")
    for (ln, c, w) in slots:
        bullets.append(f'line {ln} ({c}, {w} words) exceeds the {budget}-word slot budget -- shorten to values only (numbers, names, URLs, markers, short labels).')
    return prompt + '\n\nYOUR IR:\n' + ir + '\n\nThat IR violates the emission rules:\n' + '\n'.join('- ' + b for b in bullets) + '\n\nRe-emit the FULL corrected IR (every line, in order, with all entities preserved). FIXED IR:'


def autofix_ir(text):
    out = []
    fixed = 0
    for ln in text.split('\n'):
        s = ln.strip()
        parts = _split_code(s)
        if not parts:
            # ':'-led non-codes are malformed upstream; '@' lines are left
            # alone (spanrefs can't be judged without a store).
            if s.startswith(':'):
                out.append('Note: ' + s)
                fixed += 1
            else:
                out.append(ln)
            continue
        code, rest = parts
        if code not in ARITIES:
            t = ', '.join(rest.split('|'))
            if t.strip():
                out.append(t)
            fixed += 1
            continue
        want = ARITIES[code]
        vals = rest.split('|')
        while vals and vals[-1] == '':
            vals.pop()
        if len(vals) == want:
            out.append(ln)
            continue
        fixed += 1
        if want == 0:
            out.append(':' + code)
            t = ', '.join(vals)
            if t.strip():
                out.append(t)
            continue
        if len(vals) > want:
            tail = ', '.join(vals[want - 1:])
            out.append('|'.join([code] + vals[:want - 1] + [tail]))
            continue
        if len(vals) == 1:
            pieces = [p for p in (q.strip() for q in vals[0].split(',')) if p]
            if len(pieces) == want:
                out.append('|'.join([code] + pieces))
                continue
        t = ', '.join(vals)
        if t.strip():
            out.append(t)
    return '\n'.join(out), fixed


def chat(messages, max_tokens):
    body = json.dumps({'messages': messages, 'temperature': TEMP, 'max_tokens': max_tokens}).encode()
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/chat/completions', data=body, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)


def emit(prompt):
    d = chat([{'role': 'user', 'content': prompt}], MAX_TOKENS)
    return (d['choices'][0]['message']['content'] or ''), d['choices'][0]['finish_reason'], d


s = chat([{'role': 'user', 'content': 'Reply with exactly: ok'}], 50)
txt = s['choices'][0]['message']['content'] or ''
print('smoke:', repr(txt[:80]), s['choices'][0]['finish_reason'], flush=True)
assert txt.strip(), 'smoke empty'
hb('smoke', txt[:60])

outp = '/kaggle/working/emit-v9-gemma.jsonl'
n_err = 0
coded_total = line_total = 0
with open(outp, 'w') as out:
    for idx, item in enumerate(REFS):
        prompt = PROMPT_HEAD + '\n---\n\nREPLY TO COMPRESS:\n' + item['reference'] + '\n\nIR:'
        row = {'key': item['key'], 'irFirst': '', 'irCoded': '', 'repaired': False, 'budgetFails': 0, 'arityFails': 0, 'fixedLines': 0,
               'finish': '', 'wallMs': 0, 'compTokens': None, 'error': None}
        ok = False
        for attempt in range(2):
            try:
                t0 = time.time()
                text, finish, d = emit(prompt)
                row['wallMs'] = round((time.time() - t0) * 1000)
                usg = d.get('usage') or {}
                row['compTokens'] = usg.get('completion_tokens')
                row['finish'] = finish
                row['irFirst'] = text
                ir = text
                comp_total = usg.get('completion_tokens') or 0
                ir, n0 = autofix_ir(ir)
                row['fixedLines'] = n0
                for _r in range(2):
                    cv = code_violations(ir)
                    if not cv:
                        break
                    rtext, rfinish, rd = emit(repair_ask(prompt, ir, cv, [], BUDGET))
                    ir, n1 = autofix_ir(rtext)
                    row['fixedLines'] += n1
                    row['repaired'] = True
                    row['finish'] = rfinish
                    rusg = rd.get('usage') or {}
                    comp_total += rusg.get('completion_tokens') or 0
                if row['compTokens'] is not None:
                    row['compTokens'] = comp_total
                row['irCoded'] = ir
                row['budgetFails'] = len(slot_violations(ir, BUDGET))
                row['arityFails'] = sum(1 for _ln, _c, k, _w, _g in code_violations(ir) if k == 'arity')
                ok = True
                break
            except Exception as e:
                row['error'] = f'{type(e).__name__}: {e}'[:200]
                time.sleep(10)
        if not ok:
            n_err += 1
        out.write(json.dumps(row) + '\n')
        out.flush()
        for ln in row['irCoded'].split('\n'):
            s = ln.strip()
            if not s:
                continue
            line_total += 1
            if CODE_RE.match(s):
                coded_total += 1
        if (idx + 1) % 10 == 0:
            cu = coded_total / max(1, line_total)
            print(f'{idx + 1}/{NREFS}, {n_err} errors, codeUse {cu:.2f}', flush=True)
            hb('progress', f'{idx + 1}/{NREFS} {n_err}err')
        if idx == 9:
            cu = coded_total / max(1, line_total)
            assert cu >= 0.5, f'EMIT COMPLIANCE FAIL: codeUse {cu:.2f} at row 10'
        assert n_err <= 10, f'error budget exceeded ({n_err})'
print('done')
hb('done', 'emit-gemma complete')
